# Agents와 Structured Outputs

구조화된 출력을 통해 에이전트는 특정하고 예측 가능한 형식으로 데이터를 반환할 수 있습니다.

자연어 응답을 파싱하는 대신, 애플리케이션에서 직접 사용할 수 있는 JSON 객체, Pydantic 모델 또는 데이터 클래스 형태의 구조화된 데이터를 얻을 수 있습니다.

> https://docs.langchain.com/oss/python/langchain/structured-output

In [3]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [7]:
from pydantic import BaseModel, Field

class MovieInfo(BaseModel):
    """영화에 대한 세부 정보."""
    title: str = Field(..., description="영화의 제목")
    year: int = Field(..., description="영화가 개봉된 연도")
    director: str = Field(..., description="영화의 감독")
    rating: float = Field(..., description="10점 만점 영화 평점")

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[],
    response_format=MovieInfo
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [9]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "영화 기생충에 대한 세부 정보를 제공해줘"}]
})

result

{'messages': [HumanMessage(content='영화 기생충에 대한 세부 정보를 제공해줘', additional_kwargs={}, response_metadata={}, id='2212e0c0-5175-4123-9996-186e6833b89f'),
  AIMessage(content='{\n"title": "기생충",\n"year": 2019,\n"director": "봉준호",\n"rating": 8.5\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b9c90-63bf-7252-9ead-376c2cbe1015-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 184, 'total_tokens': 197, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 145}})],
 'structured_response': MovieInfo(title='기생충', year=2019, director='봉준호', rating=8.5)}

In [10]:
print(result["structured_response"])

title='기생충' year=2019 director='봉준호' rating=8.5


### Tool calling strategy  (도구 호출 전략)

구조화된 출력을 기본적으로 지원하지 않는 모델의 경우, LangChain은 툴 호출을 사용하여 동일한 결과를 얻습니다.

이는 툴 호출을 지원하는 모든 모델(대부분의 최신 모델)에서 작동합니다.

In [2]:
# 테스트용 모의 데이터
mock_emails = {
    101: {
        "sender": "angry_customer@gmail.com",
        "subject": "배송 지연 문의",
        "body": "안녕하세요. 지난주에 주문한 노트북 배송이 계속 지연되고 있습니다. 벌써 3일째인데 아무런 연락이 없네요. 매우 실망스럽습니다. 빠른 확인 바랍니다."
    },
    102: {
        "sender": "happy_user@naver.com",
        "subject": "업데이트 칭찬",
        "body": "새로 업데이트된 기능을 써봤는데 정말 편리하네요! 특히 다크 모드가 눈이 안 아파서 좋습니다. 개발팀에 감사드려요."
    },
    103: {
        "sender": "wondering@kakao.com",
        "subject": "환불 규정 문의",
        "body": "환불 규정이 궁금합니다. 단순 변심인 경우에도 배송비 무료인가요?"
    }
}

In [4]:
from langchain.tools import tool
import json

@tool
def read_email(email_id: int) -> str:
    """ID를 통해 이메일 상세 내용(발신자, 제목, 본문)을 읽어옵니다."""
    email_data = mock_emails.get(email_id)
    if not email_data:
        return "해당 ID의 이메일을 찾을 수 없습니다."
    
    print(f"[시스템] 이메일 ID {email_id} 읽기 완료 (발신자: {email_data['sender']})")
    # 에이전트가 구조를 잘 파악할 수 있도록 JSON 문자열로 반환합니다.
    return json.dumps(email_data, ensure_ascii=False)

@tool
def send_email(to: str, subject: str, body: str) -> bool:
    """고객에게 이메일을 직접 발송합니다."""
    print(f"\n[이메일 발송 중...]")
    print(f"받는 사람: {to}")
    print(f"제목: {subject}")
    print(f"내용: {body}")
    print(f"--------------------------------------------------\n")
    return True

In [5]:
from pydantic import BaseModel, Field
from typing import List, Literal

class CustomerServiceReport(BaseModel):
    intent: str = Field(description="고객의 핵심 의도 (예: 배송 불만, 기능 칭찬, 규정 문의 등)")
    sentiment: Literal["positive", "neutral", "negative"] = Field(description="고객의 감성 상태 분석")
    summary: str = Field(description="이메일 내용의 1줄 요약")
    required_actions: List[str] = Field(description="이 문제를 해결하기 위해 내부에서 처리해야 할 조치 리스트")
    composed_reply: str = Field(description="고객의 감성을 케어하고 상황을 해결하는 정중한 답장 본문")

In [6]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[send_email, read_email],
    response_format=ToolStrategy(CustomerServiceReport)
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [7]:
response = agent.invoke({
    "messages": [
        {
            "role": "user", 
            "content": "103번 이메일을 분석하고, 분석 내용에 기반해 해당 고객(발신자)에게 적절한 답장을 직접 보내줘."
        }
    ]
})

[시스템] 이메일 ID 103 읽기 완료 (발신자: wondering@kakao.com)

[이메일 발송 중...]
받는 사람: wondering@kakao.com
제목: [답변] 환불 규정 문의
내용: 안녕하세요, 고객님. 환불 규정에 대해 문의 주셔서 감사합니다. 단순 변심으로 인한 환불의 경우, 왕복 배송비는 고객님께서 부담하시게 됩니다. 상품을 받으신 날로부터 7일 이내에 신청하실 수 있으며, 상품이 훼손되지 않고 재판매 가능한 상태여야 합니다. 더 궁금하신 점이 있으시면 언제든지 다시 문의해주세요. 감사합니다.
--------------------------------------------------



In [8]:
response

{'messages': [HumanMessage(content='103번 이메일을 분석하고, 분석 내용에 기반해 해당 고객(발신자)에게 적절한 답장을 직접 보내줘.', additional_kwargs={}, response_metadata={}, id='9fd4b706-b21f-4445-bb34-62565f69c6d2'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"email_id": 103}'}, '__gemini_function_call_thought_signatures__': {'b66d5b96-a42f-4727-89da-c6bae2c6c746': 'CqUJAXLI2nz1uP1aBKmkVNAWvkMbdKjUvHhBHu2x2q6QYJ0ui4mZzJ6tStg0ZC1Q81idlC/E5BBZwqUH3L5XfXfc+m3d69nd0AtHIB4Kct37vY3fIlb+VK88/3XWduvhpSSwOviM7v6TlEL9a+iCUiECXumfWGY9WFPJ49E0palEZbtxuo7TEnWkMa0ULmeSX9rfB5pCMfeqaa0xyFz0VAC7PSIfP58GjTQ4lJyxAMDugba13T6ikFNZxlh5EX4sqjkPCJ+3NWC7yguGKV0Zltapbe6BIOT3JRzdlcGqwtLG18+e3CdNYh5YnrVxMuI2spdMYlt4jezHGCdGKZpKiCCP8smc+oXcwys8QRoiow25YHRGwgrQG6UQGIkavzh1oEht5FJwLpMfJRhaewuRbpwx4T8pQVMzGgrvKgA2PVNWHUDXRLOQgui0J068y0/Cbz1CnQBOSTmIhodeqeRoZZfOFuosIbLQUWDXPo9uDMbiwaQjSr/Q8gyFxvzv/g69TeVKOGeWIRpvZXBQCtkMz3zwOskq1GKDeBfg73Idv/tCOppS8UiLViC+aZ4We23mBk7SOHWaYjfbi3pkaWd8nXS09nu8mz/P8rB

In [9]:
response["structured_response"]

CustomerServiceReport(intent='환불 규정 문의', sentiment='neutral', summary='단순 변심으로 인한 환불 시 배송비 발생 여부 문의', required_actions=['고객에게 환불 규정 안내', '단순 변심 시 배송비 발생 여부 안내'], composed_reply='안녕하세요, 고객님. 환불 규정에 대해 문의 주셔서 감사합니다. 단순 변심으로 인한 환불의 경우, 왕복 배송비는 고객님께서 부담하시게 됩니다. 상품을 받으신 날로부터 7일 이내에 신청하실 수 있으며, 상품이 훼손되지 않고 재판매 가능한 상태여야 합니다. 더 궁금하신 점이 있으시면 언제든지 다시 문의해주세요. 감사합니다.')